# Notebook 6.3: Exploring Gradient Descent Visually

**Companion to Chapter 6: Simple Linear Regression**  
*Machine Learning with Python: Principles and Practical Techniques*

> **Estimated time:** 40–50 minutes  
> **Level:** Beginner to intermediate  
> **Environment:** Google Colab or Jupyter Notebook

---

## Related chapter ideas

This notebook reinforces the chapter's mathematical treatment of gradient descent through code, experimentation, and visualization. The purpose is to **see the algorithm work**, not repeat every derivation from the book.

## Learning objectives

By the end of this notebook, you will be able to:

1. explain the gradient-descent update cycle;
2. perform one simultaneous update of the intercept and slope;
3. implement batch gradient descent with NumPy;
4. visualize the regression line moving toward the data;
5. explain how the learning rate affects convergence;
6. recognize slow convergence, overshooting, and divergence; and
7. compare gradient-descent parameters with Scikit-learn's solution.


## What will you explore?

Using the same research-experience dataset as Notebooks 6.1 and 6.2, you will observe this cycle:

**Predict → Calculate cost → Calculate gradients → Update parameters → Repeat**

> **Key principle:** Gradient descent changes the parameters, not the observed data. Each update should move the fitted line toward a lower-cost position.


## 1. Import the libraries


In [ ]:
from io import StringIO

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression

pd.set_option("display.precision", 4)
print("NumPy version:", np.__version__)


## 2. Load the dataset


In [ ]:
stipend_csv = """experience_years,annual_stipend_thousands
0.5,34
1.0,37
1.5,39
2.0,43
2.5,45
3.0,49
3.5,50
4.0,54
4.5,57
5.0,60
5.5,61
6.0,66
6.5,68
7.0,70
7.5,74
8.0,77
8.5,79
9.0,82
9.5,85
10.0,88
"""

stipends = pd.read_csv(StringIO(stipend_csv))
print("Dataset shape:", stipends.shape)
display(stipends.head())


In [ ]:
X = stipends["experience_years"].to_numpy(dtype=float)
y = stipends["annual_stipend_thousands"].to_numpy(dtype=float)
n = len(X)

display(stipends.head())
print("Number of observations:", n)


## 3. Recall the model and cost


The hypothesis is:

$$h_\theta(x)=\theta_0+\theta_1x$$

The cost function is:

$$J(\theta_0,\theta_1)=\frac{1}{2n}\sum_{i=1}^{n}\left(h_\theta(x_i)-y_i\right)^2$$

Gradient descent repeats these simultaneous updates:

$$\theta_0 := \theta_0-\alpha\frac{1}{n}\sum_{i=1}^{n}(h_\theta(x_i)-y_i)$$

$$\theta_1 := \theta_1-\alpha\frac{1}{n}\sum_{i=1}^{n}(h_\theta(x_i)-y_i)x_i$$

Here, $\alpha$ is the learning rate.


In [ ]:
def predict(x_values, theta_0, theta_1):
    return theta_0 + theta_1 * np.asarray(x_values)


def cost_function(x_values, y_values, theta_0, theta_1):
    errors = predict(x_values, theta_0, theta_1) - y_values
    return np.mean(errors ** 2) / 2


def gradients(x_values, y_values, theta_0, theta_1):
    errors = predict(x_values, theta_0, theta_1) - y_values
    return np.mean(errors), np.mean(errors * x_values)


print("Initial cost at θ₀=0 and θ₁=0:", round(cost_function(X, y, 0, 0), 3))


## 4. Perform one update


In [ ]:
theta_0 = 0.0
theta_1 = 0.0
alpha = 0.01

gradient_0, gradient_1 = gradients(X, y, theta_0, theta_1)

# Calculate both new values from the same old parameter values.
new_theta_0 = theta_0 - alpha * gradient_0
new_theta_1 = theta_1 - alpha * gradient_1

one_step = pd.DataFrame({
    "parameter": ["θ₀", "θ₁"],
    "old_value": [theta_0, theta_1],
    "gradient": [gradient_0, gradient_1],
    "new_value": [new_theta_0, new_theta_1],
})

display(one_step)
print("Cost before:", round(cost_function(X, y, theta_0, theta_1), 3))
print("Cost after:", round(cost_function(X, y, new_theta_0, new_theta_1), 3))


Both updates use gradients computed from the same old parameter state. Updating $\theta_0$ first and then using its new value to calculate the $\theta_1$ update would not follow the derived simultaneous-update rule.


## 5. Implement gradient descent


In [ ]:
def gradient_descent(
    x_values,
    y_values,
    alpha=0.01,
    iterations=3000,
    theta_0=0.0,
    theta_1=0.0,
):
    history = []

    for iteration in range(iterations + 1):
        current_cost = cost_function(x_values, y_values, theta_0, theta_1)
        history.append((iteration, theta_0, theta_1, current_cost))

        if iteration == iterations or not np.isfinite(current_cost):
            break

        gradient_0, gradient_1 = gradients(
            x_values, y_values, theta_0, theta_1
        )
        next_theta_0 = theta_0 - alpha * gradient_0
        next_theta_1 = theta_1 - alpha * gradient_1
        theta_0, theta_1 = next_theta_0, next_theta_1

    history = pd.DataFrame(
        history, columns=["iteration", "theta_0", "theta_1", "cost"]
    )
    return theta_0, theta_1, history


In [ ]:
gd_intercept, gd_slope, history = gradient_descent(
    X, y, alpha=0.01, iterations=5000
)

print("Gradient-descent intercept:", round(gd_intercept, 4))
print("Gradient-descent slope:", round(gd_slope, 4))
print("Final cost:", round(history.iloc[-1]["cost"], 6))
display(history.iloc[[0, 1, 10, 100, 1000, -1]])


## 6. Watch cost decrease


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

axes[0].plot(history["iteration"], history["cost"], color="#4C78A8")
axes[0].set_title("Cost During Training")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Cost J")
axes[0].grid(alpha=0.25)

axes[1].plot(history["iteration"], history["cost"], color="#4C78A8")
axes[1].set_yscale("log")
axes[1].set_title("Cost on a Logarithmic Scale")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Cost J")
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()


The cost falls rapidly at first and then improves more slowly. A logarithmic scale makes later changes visible. A healthy run should generally move downward rather than oscillate or explode.


## 7. Watch the line move toward the data


In [ ]:
snapshots = [0, 1, 5, 20, 100, 1000, 5000]
selected = history.loc[history["iteration"].isin(snapshots)]

fig, axes = plt.subplots(2, 4, figsize=(14, 7), sharex=True, sharey=True)
axes = axes.ravel()

for axis, (_, row) in zip(axes, selected.iterrows()):
    axis.scatter(X, y, color="#4C78A8", edgecolor="black", s=35)
    axis.plot(X, predict(X, row["theta_0"], row["theta_1"]),
              color="#E45756", linewidth=2)
    axis.set_title(f"Iteration {int(row['iteration'])}\nJ={row['cost']:.3f}")
    axis.grid(alpha=0.15)

for axis in axes[len(selected):]:
    axis.axis("off")

fig.supxlabel("Research experience (years)")
fig.supylabel("Annual stipend ($ thousands)")
plt.tight_layout()
plt.show()


Each panel is a snapshot of the same algorithm. The data remain fixed while the intercept and slope move the line toward a lower-cost position.


## 8. See the optimization path


In [ ]:
theta_0_values = np.linspace(20, 42, 100)
theta_1_values = np.linspace(3, 8, 100)
T0, T1 = np.meshgrid(theta_0_values, theta_1_values)
J = np.empty_like(T0)

for row in range(T0.shape[0]):
    for column in range(T0.shape[1]):
        J[row, column] = cost_function(X, y, T0[row, column], T1[row, column])

fig, ax = plt.subplots(figsize=(7, 5.5))
contours = ax.contour(T0, T1, J, levels=25, cmap="viridis")
path = history.iloc[::100]
ax.plot(path["theta_0"], path["theta_1"], color="#E45756",
        marker="o", markersize=3, linewidth=1.5, label="Update path")
ax.scatter(gd_intercept, gd_slope, color="black", marker="*", s=140,
           label="Final parameters", zorder=5)
ax.set_title("Gradient Descent on Cost Contours")
ax.set_xlabel("Intercept θ₀")
ax.set_ylabel("Slope θ₁")
ax.legend()
plt.tight_layout()
plt.show()


Each contour joins parameter combinations with the same cost. The path crosses contours toward the single global minimum of this convex simple-linear-regression problem.


## 9. Experiment with the learning rate


In [ ]:
learning_rates = [0.0001, 0.001, 0.01, 0.03]
rate_histories = {}
rate_results = []

for rate in learning_rates:
    intercept, slope, rate_history = gradient_descent(
        X, y, alpha=rate, iterations=1000
    )
    rate_histories[rate] = rate_history
    final_cost = rate_history.iloc[-1]["cost"]
    rate_results.append({
        "learning_rate": rate,
        "iterations_completed": len(rate_history) - 1,
        "final_cost": final_cost,
        "finite": np.isfinite(final_cost),
    })

display(pd.DataFrame(rate_results))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for rate, rate_history in rate_histories.items():
    finite = rate_history.loc[np.isfinite(rate_history["cost"])]
    ax.plot(finite["iteration"], finite["cost"], label=f"α={rate}")

ax.set_yscale("log")
ax.set_title("Effect of the Learning Rate")
ax.set_xlabel("Iteration")
ax.set_ylabel("Cost J (log scale)")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


- **Too small:** Cost decreases, but progress is slow.
- **Appropriate:** Cost decreases steadily and efficiently.
- **Too large:** Updates overshoot the minimum and may diverge.

There is no universally correct learning rate. A useful value depends on feature scales and the shape of the cost surface.


## 10. Demonstrate an excessively large learning rate


In [ ]:
_, _, unstable_history = gradient_descent(
    X, y, alpha=0.1, iterations=50
)

display(unstable_history.head(10))
print("Last recorded cost:", unstable_history.iloc[-1]["cost"])


The rapidly increasing values indicate divergence. In practice, stop such a run and reduce the learning rate; do not interpret the overflowing parameter values.


## 11. Compare with Scikit-learn


In [ ]:
sklearn_model = LinearRegression()
sklearn_model.fit(X.reshape(-1, 1), y)

comparison = pd.DataFrame({
    "method": ["Gradient descent", "Scikit-learn LinearRegression"],
    "intercept": [gd_intercept, sklearn_model.intercept_],
    "slope": [gd_slope, sklearn_model.coef_[0]],
    "cost": [
        cost_function(X, y, gd_intercept, gd_slope),
        cost_function(X, y, sklearn_model.intercept_, sklearn_model.coef_[0]),
    ],
})

display(comparison)


The parameters are nearly identical. Scikit-learn's `LinearRegression` computes an ordinary least-squares solution directly; this gradient-descent implementation reaches the same minimum iteratively.

The purpose of implementing gradient descent here is conceptual understanding—not replacing the tested library implementation used in Notebook 6.2.


## 12. Try a new starting point


In [ ]:
start_intercept = 50.0
start_slope = 1.0

different_intercept, different_slope, different_history = gradient_descent(
    X, y, alpha=0.01, iterations=5000,
    theta_0=start_intercept, theta_1=start_slope,
)

print("Starting parameters:", start_intercept, start_slope)
print("Final parameters:", round(different_intercept, 4), round(different_slope, 4))
print("Final cost:", round(different_history.iloc[-1]["cost"], 6))


For this convex cost surface, reasonable starting points converge toward the same global minimum when the learning rate is stable and enough iterations are allowed.


## 13. Guided practice


1. Calculate both gradients at $\theta_0=30$ and $\theta_1=5$.
2. Perform one simultaneous update using $\alpha=0.01$.
3. Run 500 iterations with $\alpha=0.001$ and report the final cost.
4. Compare the gradient magnitude at initialization and after 5,000 iterations.
5. Explain why a large gradient does not automatically mean the model is wrong.


In [ ]:
# Write your solution here.


<details>
<summary><strong>Open the suggested solution</strong></summary>

```python
# 1. Gradients
g0, g1 = gradients(X, y, 30, 5)
print(g0, g1)

# 2. One update
updated_t0 = 30 - 0.01 * g0
updated_t1 = 5 - 0.01 * g1
print(updated_t0, updated_t1)

# 3. Smaller learning rate
_, _, practice_history = gradient_descent(X, y, alpha=0.001, iterations=500)
print(practice_history.iloc[-1]["cost"])

# 4. Gradient magnitudes
initial_magnitude = np.linalg.norm(gradients(X, y, 0, 0))
final_magnitude = np.linalg.norm(gradients(X, y, gd_intercept, gd_slope))
print(initial_magnitude, final_magnitude)

# 5. A large gradient indicates a steep local cost surface; it often occurs when
# parameters are far from the minimum or features have a large numerical scale.
```

</details>


## 14. Challenge: Recommend a learning rate


Test at least five learning rates between `0.00001` and `0.05`. Use the same starting point and iteration budget for each run.

Prepare a small recommendation containing:

1. final cost;
2. whether the run remained finite;
3. how quickly cost fell;
4. whether the history oscillated; and
5. your recommended learning rate with evidence.

Do not select a rate only because it gives the lowest cost after one update; examine the full trajectory.


## 15. Common mistakes to avoid


| Mistake | Consequence | Better practice |
|---|---|---|
| Updating parameters sequentially | Uses inconsistent parameter states | Calculate both new values before assignment |
| Using a very large learning rate | Overshooting or divergence | Monitor cost and reduce $\alpha$ |
| Using too few iterations | Stops far from the minimum | Inspect the convergence curve |
| Ignoring numerical scale | Stable learning-rate range may become narrow | Standardize when appropriate |
| Treating one learning rate as universal | Different problems have different surfaces | Experiment systematically |
| Using test data during optimization | Leaks evaluation information | Optimize with training data only |


## 16. Reflection


1. Why does subtracting the gradient move parameters downhill?
2. Why must the parameter updates be simultaneous?
3. What visual evidence indicates convergence?
4. How do small and large learning rates behave differently?
5. Why can different starting points reach the same solution here?
6. Why should practitioners usually use Scikit-learn rather than this teaching implementation?


## 17. Key takeaways


- Gradient descent repeatedly updates parameters to reduce the cost function.
- The gradient provides direction; the learning rate controls step size.
- Intercept and slope updates must use the same old parameter values.
- Cost history reveals slow convergence, oscillation, or divergence.
- The fitted line visibly moves toward a lower-error position as training proceeds.
- Simple linear regression has a convex squared-error surface with one global minimum.
- The from-scratch result agrees with Scikit-learn, connecting optimization theory with practical implementation.

## Chapter 6 notebook journey complete

1. **Understand the regression line, residuals, and cost**
2. **Implement and evaluate SLR with Scikit-learn**
3. **Explore gradient descent through visual experiments**
